# Task 4 - CNN Pipeline
This notebook documents the completed Task 4. The implementation starts from the Task 3 CNN, adds training-only augmentation, retrains the model, logs repeated experiments, and compares the updated CNN against earlier saved ones.



## Theory

### Why does training on a limited dataset lead to overfitting, and how does augmentation help?
A limited dataset leads to overfitting because it gives to few examples. Because, instead of learning general patterns, the model may memorize these and perform extremely well on the dataset, but horribly when new information is introduced. Augmentation can help by creating different variations of the same images, improving generalization and therefore model performance in the longterm.

### What transformations are commonly applied to image data, and when is each appropriate?
The most common transformations are flips, rotations, crops, zooming, blurring, color jittering, and each may be the most effective when it maintains the class of the image. For example, flips and small rotations may be helpful when object orientation doesn't change the label, whilst color jittering is especially useful with varying lighting conditions.

### How does augmentation interact with label structure, especially for pixel-level prediction tasks?
The image and label must be transformed together so that the pixel alignment is maintained, this is because when transforming an image using crops, flips or another method it must be identically applied to both so that the augmentation fully works.


In [376]:
from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.errors import NotGeoreferencedWarning

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)


In [ ]:


# Configuration And Paths

def resolve_task_dir(start_dir):
    candidates = [
        start_dir,
        start_dir / "Task_4",
        start_dir / "Learning_Path" / "Task_4",
    ]

    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "CNN.py").exists():
            return candidate

    raise RuntimeError(
        "Could not locate Task_4. Run the notebook from Learning_Path or Learning_Path/Task_4."
    )

current_dir = Path.cwd().resolve()
TASK_DIR = resolve_task_dir(current_dir)

BASE_DIR = TASK_DIR.parent
SAMPLES_DIR = BASE_DIR / "samples"
LABELS_DIR = BASE_DIR / "labels"
OUTPUT_DIR = TASK_DIR / "outputs"
LATEST_RESULTS_PATH = OUTPUT_DIR / "basic_results.txt"
RESULTS_LOG_PATH = OUTPUT_DIR / "experiment_log.json"
RESULTS_SUMMARY_PATH = OUTPUT_DIR / "experiment_summary.md"

# Update this before each run so the saved history stays readable.
EXPERIMENT_NAME = "CNNJ baseline | augmentation | ReLU | Adam | lr=0.001 | epochs=30 | kernel=5x5"
RUN_NOTES = ""

IMAGE_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 0.0005
PATIENCE = 10
SEED = 42

LOW_THRESHOLD = 150
MEDIUM_THRESHOLD = 180
CLASS_NAMES = ["low", "medium", "high"]

def set_random_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Current working directory: {current_dir}")
print(f"Task directory: {TASK_DIR}")
print(f"Running on: {device}")

# Data Helper Functions

def label_to_class(label_tile):
    mean_ndvi = float(np.nanmean(label_tile))

    if mean_ndvi < LOW_THRESHOLD:
        return 0
    if mean_ndvi < MEDIUM_THRESHOLD:
        return 1
    return 2

def load_dataset(samples_dir, labels_dir, image_size):
    images = []
    labels = []

    for sample_path in sorted(samples_dir.glob("*.tif*")):
        label_path = labels_dir / sample_path.name.replace("img_", "ndvi_")

        if not label_path.exists():
            continue

        with rasterio.open(sample_path) as src:
            image = src.read(
                out_shape=(src.count, image_size, image_size),
                resampling=Resampling.bilinear,
            ).astype(np.float32)

        with rasterio.open(label_path) as src:
            label = src.read(1).astype(np.float32)

        image = image / 255.0
        image = np.nan_to_num(image)

        images.append(image)
        labels.append(label_to_class(label))

    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int64)

    if len(images) == 0:
        raise RuntimeError("No image/label pairs were loaded.")

    return images, labels

def print_dataset_summary(images, labels):
    print(f"Total images: {len(images)}")
    print(f"Class counts: {dict(zip(CLASS_NAMES, np.bincount(labels, minlength=3)))}")

def split_dataset(images, labels, seed):
    train_images, temp_images, train_labels, temp_labels = train_test_split(
        images,
        labels,
        test_size=0.30,
        stratify=labels,
        random_state=seed,
    )

    val_images, test_images, val_labels, test_labels = train_test_split(
        temp_images,
        temp_labels,
        test_size=0.50,
        stratify=temp_labels,
        random_state=seed,
    )

    print(f"Train: {len(train_images)}")
    print(f"Validation: {len(val_images)}")
    print(f"Test: {len(test_images)}")

    return (
        train_images,
        val_images,
        test_images,
        train_labels,
        val_labels,
        test_labels,
    )

def compute_normalization_stats(train_images):
    mean = train_images.mean(axis=(0, 2, 3), keepdims=True)
    std = train_images.std(axis=(0, 2, 3), keepdims=True)
    std[std == 0] = 1.0
    return mean, std

def normalize_images(images, mean, std):
    return (images - mean) / std


Current working directory: /Users/jozuas/Desktop/Research - PandaHat/Learning_Path/Task_4
Task directory: /Users/jozuas/Desktop/Research - PandaHat/Learning_Path/Task_4
Running on: cpu


In [378]:


# Configuration And Paths

def resolve_task_dir(start_dir):
    candidates = [
        start_dir,
        start_dir / "Task_4",
        start_dir / "Learning_Path" / "Task_4",
    ]

    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "CNN.py").exists():
            return candidate

    raise RuntimeError(
        "Could not locate Task_4. Run the notebook from Learning_Path or Learning_Path/Task_4."
    )

current_dir = Path.cwd().resolve()
TASK_DIR = resolve_task_dir(current_dir)

BASE_DIR = TASK_DIR.parent
SAMPLES_DIR = BASE_DIR / "samples"
LABELS_DIR = BASE_DIR / "labels"
OUTPUT_DIR = TASK_DIR / "outputs"
LATEST_RESULTS_PATH = OUTPUT_DIR / "basic_results.txt"
RESULTS_LOG_PATH = OUTPUT_DIR / "experiment_log.json"
RESULTS_SUMMARY_PATH = OUTPUT_DIR / "experiment_summary.md"

# Update this before each run so the saved history stays readable.
EXPERIMENT_NAME = "CNNJ baseline | augmentation | ReLU | Adam | lr=0.001 | epochs=30 | kernel=5x5"
RUN_NOTES = ""

IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 30
LEARNING_RATE = 0.0005
PATIENCE = 10
SEED = 42

LOW_THRESHOLD = 150
MEDIUM_THRESHOLD = 180
CLASS_NAMES = ["low", "medium", "high"]

def set_random_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Current working directory: {current_dir}")
print(f"Task directory: {TASK_DIR}")
print(f"Running on: {device}")

# Data Helper Functions

def label_to_class(label_tile):
    mean_ndvi = float(np.nanmean(label_tile))

    if mean_ndvi < LOW_THRESHOLD:
        return 0
    if mean_ndvi < MEDIUM_THRESHOLD:
        return 1
    return 2

def load_dataset(samples_dir, labels_dir, image_size):
    images = []
    labels = []

    for sample_path in sorted(samples_dir.glob("*.tif*")):
        label_path = labels_dir / sample_path.name.replace("img_", "ndvi_")

        if not label_path.exists():
            continue

        with rasterio.open(sample_path) as src:
            image = src.read(
                out_shape=(src.count, image_size, image_size),
                resampling=Resampling.bilinear,
            ).astype(np.float32)

        with rasterio.open(label_path) as src:
            label = src.read(1).astype(np.float32)

        image = image / 255.0
        image = np.nan_to_num(image)

        images.append(image)
        labels.append(label_to_class(label))

    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int64)

    if len(images) == 0:
        raise RuntimeError("No image/label pairs were loaded.")

    return images, labels

def print_dataset_summary(images, labels):
    print(f"Total images: {len(images)}")
    print(f"Class counts: {dict(zip(CLASS_NAMES, np.bincount(labels, minlength=3)))}")

def split_dataset(images, labels, seed):
    train_images, temp_images, train_labels, temp_labels = train_test_split(
        images,
        labels,
        test_size=0.30,
        stratify=labels,
        random_state=seed,
    )

    val_images, test_images, val_labels, test_labels = train_test_split(
        temp_images,
        temp_labels,
        test_size=0.50,
        stratify=temp_labels,
        random_state=seed,
    )

    print(f"Train: {len(train_images)}")
    print(f"Validation: {len(val_images)}")
    print(f"Test: {len(test_images)}")

    return (
        train_images,
        val_images,
        test_images,
        train_labels,
        val_labels,
        test_labels,
    )

def compute_normalization_stats(train_images):
    mean = train_images.mean(axis=(0, 2, 3), keepdims=True)
    std = train_images.std(axis=(0, 2, 3), keepdims=True)
    std[std == 0] = 1.0
    return mean, std

def normalize_images(images, mean, std):
    return (images - mean) / std


Current working directory: /Users/jozuas/Desktop/Research - PandaHat/Learning_Path/Task_4
Task directory: /Users/jozuas/Desktop/Research - PandaHat/Learning_Path/Task_4
Running on: cpu


In [379]:
def label_to_class(label_tile):
    mean_ndvi = float(np.nanmean(label_tile))

    if mean_ndvi < LOW_THRESHOLD:
        return 0
    if mean_ndvi < MEDIUM_THRESHOLD:
        return 1
    return 2


# -----------------------------------------------------------------------------
# Load the TIFF data
# -----------------------------------------------------------------------------

images, labels = load_dataset(SAMPLES_DIR, LABELS_DIR, IMAGE_SIZE)
print_dataset_summary(images, labels)


Total images: 614
Class counts: {'low': np.int64(27), 'medium': np.int64(338), 'high': np.int64(249)}


In [380]:
# -----------------------------------------------------------------------------
# Split into train / validation / test
# -----------------------------------------------------------------------------

(
    train_images,
    val_images,
    test_images,
    train_labels,
    val_labels,
    test_labels,
) = split_dataset(images, labels, SEED)

mean, std = compute_normalization_stats(train_images)
train_images = normalize_images(train_images, mean, std)
val_images = normalize_images(val_images, mean, std)
test_images = normalize_images(test_images, mean, std)


Train: 429
Validation: 92
Test: 93


In [381]:
# -----------------------------------------------------------------------------
# Data Augmentation (only for training) with pytorch 
# -----------------------------------------------------------------------------

train_transforms = None

# torchvision is only needed if you enable augmentation below.
# If this import fails, leave train_transforms as None and continue.
# from torchvision import transforms

# Uncomment to enable training-only augmentation.
train_transforms = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    # transforms.RandomVerticalFlip(1),  # bad
    # transforms.ColorJitter(brightness=0.05, contrast=0.05), # really bad
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(1.0, 1.0)),
])


In [382]:
# Custom Dataset class to apply transforms
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

def create_dataloaders(
    train_images,
    val_images,
    test_images,
    train_labels,
    val_labels,
    test_labels,
    batch_size,
    train_transform,
):
    train_data = AugmentedDataset(train_images, train_labels, transform=train_transform)
    val_data = AugmentedDataset(val_images, val_labels, transform=None)
    test_data = AugmentedDataset(test_images, test_labels, transform=None)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_data, val_data, test_data, train_loader, val_loader, test_loader

train_data, val_data, test_data, train_loader, val_loader, test_loader = create_dataloaders(
    train_images,
    val_images,
    test_images,
    train_labels,
    val_labels,
    test_labels,
    BATCH_SIZE,
    train_transforms,
)


# -----------------------------------------------------------------------------
# CNN model
# -----------------------------------------------------------------------------

class VegetationCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16) 
        
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
         
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
           
        self.conv4 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)
        
        self.pool = nn.MaxPool2d(2,2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(256, 32)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 3)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.relu(self.bn5(self.conv5(x)))
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = OUTPUT_DIR / "basic_cnn.pth"

model = VegetationCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)


In [383]:
# Verify augmentation is active on training samples across multiple trials
num_trials = 10
non_zero_orig_aug = 0
non_zero_aug_aug = 0

for i in range(num_trials):
    orig = torch.tensor(train_images[0], dtype=torch.float32)
    aug1, _ = train_data[0]
    aug2, _ = train_data[0]

    diff_orig_aug1 = (orig - aug1).abs().mean().item()
    diff_aug1_aug2 = (aug1 - aug2).abs().mean().item()

    if diff_orig_aug1 > 0:
        non_zero_orig_aug += 1
    if diff_aug1_aug2 > 0:
        non_zero_aug_aug += 1

    print(f"Trial {i + 1:02d}: orig->aug1={diff_orig_aug1:.6f}, aug1->aug2={diff_aug1_aug2:.6f}")

print("\nSummary")
print(f"Non-zero orig->aug1 in {non_zero_orig_aug}/{num_trials} trials")
print(f"Non-zero aug1->aug2 in {non_zero_aug_aug}/{num_trials} trials")

if non_zero_orig_aug == 0 and non_zero_aug_aug == 0:
    print("Augmentation appears inactive for this sample.")
else:
    print("Augmentation is active.")


Trial 01: orig->aug1=0.256182, aug1->aug2=0.058354
Trial 02: orig->aug1=0.301598, aug1->aug2=0.334421
Trial 03: orig->aug1=0.269238, aug1->aug2=0.362253
Trial 04: orig->aug1=0.261466, aug1->aug2=0.217373
Trial 05: orig->aug1=0.364331, aug1->aug2=0.347623
Trial 06: orig->aug1=0.231965, aug1->aug2=0.327892
Trial 07: orig->aug1=0.290132, aug1->aug2=0.346466
Trial 08: orig->aug1=0.143813, aug1->aug2=0.368743
Trial 09: orig->aug1=0.363854, aug1->aug2=0.217892
Trial 10: orig->aug1=0.000000, aug1->aug2=0.306972

Summary
Non-zero orig->aug1 in 9/10 trials
Non-zero aug1->aug2 in 10/10 trials
Augmentation is active.


In [384]:
# -----------------------------------------------------------------------------
# Train
# -----------------------------------------------------------------------------

def train_one_epoch(model, loader, loss_fn, optimizer, runtime_device):
    model.train()
    running_loss = 0.0

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(runtime_device)
        batch_labels = batch_labels.to(runtime_device)

        optimizer.zero_grad()
        outputs = model(batch_images)
        loss = loss_fn(outputs, batch_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

def validate_model(model, loader, loss_fn, runtime_device):
    model.eval()
    val_loss_total = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(runtime_device)
            batch_labels = batch_labels.to(runtime_device)

            outputs = model(batch_images)
            loss = loss_fn(outputs, batch_labels)
            val_loss_total += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()

    val_loss = val_loss_total / len(loader)
    val_accuracy = correct / total
    return val_loss, val_accuracy

def train_model(
    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer,
    runtime_device,
    epochs,
    patience,
    model_path,
):
    best_val_loss = float("inf")
    best_val_accuracy = 0.0
    best_epoch = 0
    completed_epochs = 0
    wait = 0

    for epoch in range(epochs):
        completed_epochs = epoch + 1
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, runtime_device)
        val_loss, val_accuracy = validate_model(model, val_loader, loss_fn, runtime_device)

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_accuracy = val_accuracy
            best_epoch = epoch + 1
            wait = 0
            torch.save(model.state_dict(), model_path)
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break

    return {
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_accuracy": best_val_accuracy,
        "completed_epochs": completed_epochs,
    }


training_summary = train_model(
    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer,
    device,
    EPOCHS,
    PATIENCE,
    best_model_path,
)


Epoch 01/30 | Train Loss: 0.7564 | Val Loss: 1.0324 | Val Acc: 0.6087
Epoch 02/30 | Train Loss: 0.6589 | Val Loss: 0.6866 | Val Acc: 0.6848
Epoch 03/30 | Train Loss: 0.6126 | Val Loss: 0.7155 | Val Acc: 0.6848
Epoch 04/30 | Train Loss: 0.6252 | Val Loss: 0.6865 | Val Acc: 0.6957
Epoch 05/30 | Train Loss: 0.5640 | Val Loss: 0.7151 | Val Acc: 0.6848
Epoch 06/30 | Train Loss: 0.5633 | Val Loss: 0.7888 | Val Acc: 0.6848
Epoch 07/30 | Train Loss: 0.5866 | Val Loss: 0.6066 | Val Acc: 0.7500
Epoch 08/30 | Train Loss: 0.5731 | Val Loss: 0.5851 | Val Acc: 0.6957
Epoch 09/30 | Train Loss: 0.5400 | Val Loss: 0.7797 | Val Acc: 0.6630
Epoch 10/30 | Train Loss: 0.5721 | Val Loss: 0.7024 | Val Acc: 0.7174
Epoch 11/30 | Train Loss: 0.5216 | Val Loss: 0.5542 | Val Acc: 0.7174
Epoch 12/30 | Train Loss: 0.4994 | Val Loss: 0.6668 | Val Acc: 0.6957
Epoch 13/30 | Train Loss: 0.5804 | Val Loss: 0.8247 | Val Acc: 0.6522
Epoch 14/30 | Train Loss: 0.5001 | Val Loss: 0.5607 | Val Acc: 0.7609
Epoch 15/30 | Train 

KeyboardInterrupt: 

In [ ]:
# -----------------------------------------------------------------------------
# Test
# -----------------------------------------------------------------------------

def evaluate_model(model, loader, runtime_device):
    model.eval()
    all_true = []
    all_pred = []

    with torch.no_grad():
        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(runtime_device)

            outputs = model(batch_images)
            _, predicted = torch.max(outputs, 1)

            all_true.extend(batch_labels.numpy())
            all_pred.extend(predicted.cpu().numpy())

    return {
        "accuracy": accuracy_score(all_true, all_pred),
        "precision": precision_score(all_true, all_pred, average="macro", zero_division=0),
        "recall": recall_score(all_true, all_pred, average="macro", zero_division=0),
        "f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
    }

def print_test_results(metrics):
    print("\nTest Results")
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1-Score : {metrics['f1']:.4f}")

def normalize_experiment_entry(entry, index):
    default_entry = {
        "accuracy": 0.0,
        "precision": 0.0,
        "recall": 0.0,
        "f1": 0.0,
        "best_epoch": 0,
        "best_val_loss": 0.0,
        "best_val_accuracy": 0.0,
    }

    if not isinstance(entry, dict):
        return default_entry.copy()

    normalized = default_entry.copy()
    normalized.update(
        {
            key: entry[key]
            for key in default_entry
            if key in entry
        }
    )
    return normalized

def load_experiment_history(log_path):
    if not log_path.exists():
        return []

    raw_history = json.loads(log_path.read_text())
    return [
        normalize_experiment_entry(entry, index)
        for index, entry in enumerate(raw_history, start=1)
    ]

def save_experiment_history(log_path, history):
    normalized_history = [
        normalize_experiment_entry(entry, index)
        for index, entry in enumerate(history, start=1)
    ]
    log_path.write_text(json.dumps(normalized_history, indent=2))

def sort_experiment_history(history):
    normalized_history = [
        normalize_experiment_entry(entry, index)
        for index, entry in enumerate(history, start=1)
    ]

    return sorted(
        normalized_history,
        key=lambda entry: (
            entry["f1"],
            entry["accuracy"],
            entry["precision"],
            entry["recall"],
        ),
        reverse=True,
    )

def print_result_block(title, entry):
    print(f"\n{title}")
    print(f"Accuracy      : {entry['accuracy']:.4f}")
    print(f"Precision     : {entry['precision']:.4f}")
    print(f"Recall        : {entry['recall']:.4f}")
    print(f"F1-Score      : {entry['f1']:.4f}")
    print(f"Best Epoch    : {entry['best_epoch']}")
    print(f"Best Val Loss : {entry['best_val_loss']:.4f}")
    print(f"Best Val Acc  : {entry['best_val_accuracy']:.4f}")

def print_ranked_summary(ranked_results):
    print("\nResults Ordered From Best To Worst")
    print("(ranked by F1, then accuracy, precision, and recall)")

    for index, entry in enumerate(ranked_results, start=1):
        print(
            f"{index:02d}. Acc={entry['accuracy']:.4f} | "
            f"Prec={entry['precision']:.4f} | "
            f"Rec={entry['recall']:.4f} | "
            f"F1={entry['f1']:.4f} | "
            f"Best Epoch={entry['best_epoch']} | "
            f"Best Val Loss={entry['best_val_loss']:.4f} | "
            f"Best Val Acc={entry['best_val_accuracy']:.4f}"
        )

def write_summary_report(summary_path, latest_entry, ranked_results):
    best_entry = ranked_results[0]
    lines = [
        "# Task 4 Experiment Summary",
        "",
        "## Latest Test Result",
        f"- Accuracy: {latest_entry['accuracy']:.4f}",
        f"- Precision: {latest_entry['precision']:.4f}",
        f"- Recall: {latest_entry['recall']:.4f}",
        f"- F1-Score: {latest_entry['f1']:.4f}",
        f"- Best Epoch: {latest_entry['best_epoch']}",
        f"- Best Validation Loss: {latest_entry['best_val_loss']:.4f}",
        f"- Best Validation Accuracy: {latest_entry['best_val_accuracy']:.4f}",
        "",
        "## Best Result",
        f"- Accuracy: {best_entry['accuracy']:.4f}",
        f"- Precision: {best_entry['precision']:.4f}",
        f"- Recall: {best_entry['recall']:.4f}",
        f"- F1-Score: {best_entry['f1']:.4f}",
        f"- Best Epoch: {best_entry['best_epoch']}",
        f"- Best Validation Loss: {best_entry['best_val_loss']:.4f}",
        f"- Best Validation Accuracy: {best_entry['best_val_accuracy']:.4f}",
        "",
        "## Results Ordered From Best To Worst",
        "",
        "| Rank | Accuracy | Precision | Recall | F1-Score | Best Epoch | Best Val Loss | Best Val Acc |",
        "|---|---|---|---|---|---|---|---|",
    ]

    for index, entry in enumerate(ranked_results, start=1):
        lines.append(
            f"| {index} | {entry['accuracy']:.4f} | {entry['precision']:.4f} | {entry['recall']:.4f} | {entry['f1']:.4f} | {entry['best_epoch']} | {entry['best_val_loss']:.4f} | {entry['best_val_accuracy']:.4f} |"
        )

    summary_path.write_text("\n".join(lines) + "\n")

# Evaluate The Best Checkpoint

best_model = VegetationCNN().to(device)
best_model.load_state_dict(
    torch.load(best_model_path, map_location=device, weights_only=True)
)

metrics = evaluate_model(best_model, test_loader, device)
latest_result = {
    "accuracy": metrics["accuracy"],
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1": metrics["f1"],
    "best_epoch": training_summary["best_epoch"],
    "best_val_loss": training_summary["best_val_loss"],
    "best_val_accuracy": training_summary["best_val_accuracy"],
}

history = load_experiment_history(RESULTS_LOG_PATH)
history.append(latest_result)
save_experiment_history(RESULTS_LOG_PATH, history)

ranked_results = sort_experiment_history(history)
write_summary_report(RESULTS_SUMMARY_PATH, latest_result, ranked_results)

latest_lines = [
    f"Accuracy : {metrics['accuracy']:.4f}",
    f"Precision: {metrics['precision']:.4f}",
    f"Recall   : {metrics['recall']:.4f}",
    f"F1-Score : {metrics['f1']:.4f}",
    f"Best Epoch: {training_summary['best_epoch']}",
    f"Best Val Loss: {training_summary['best_val_loss']:.4f}",
    f"Best Val Acc : {training_summary['best_val_accuracy']:.4f}",
]

LATEST_RESULTS_PATH.write_text("\n".join(latest_lines) + "\n")

print_result_block("Latest Test Result", latest_result)
print_result_block("Best Result So Far", ranked_results[0])
print(f"\nTotal Logged Runs: {len(history)}")
print_ranked_summary(ranked_results)

print(f"\nSaved model: {best_model_path}")



Latest Test Result
Accuracy      : 0.8280
Precision     : 0.8024
Recall        : 0.8052
F1-Score      : 0.8035
Best Epoch    : 25
Best Val Loss : 0.5053
Best Val Acc  : 0.7826

Best Result So Far
Accuracy      : 0.8280
Precision     : 0.8024
Recall        : 0.8052
F1-Score      : 0.8035
Best Epoch    : 25
Best Val Loss : 0.5053
Best Val Acc  : 0.7826

Total Logged Runs: 32

Results Ordered From Best To Worst
(ranked by F1, then accuracy, precision, and recall)
01. Acc=0.8280 | Prec=0.8024 | Rec=0.8052 | F1=0.8035 | Best Epoch=25 | Best Val Loss=0.5053 | Best Val Acc=0.7826
02. Acc=0.8710 | Prec=0.9188 | Rec=0.6778 | F1=0.7219 | Best Epoch=25 | Best Val Loss=0.4880 | Best Val Acc=0.7826
03. Acc=0.8172 | Prec=0.8769 | Rec=0.6428 | F1=0.6846 | Best Epoch=9 | Best Val Loss=0.4960 | Best Val Acc=0.7935
04. Acc=0.8065 | Prec=0.8687 | Rec=0.6363 | F1=0.6774 | Best Epoch=28 | Best Val Loss=0.5356 | Best Val Acc=0.7283
05. Acc=0.8065 | Prec=0.8703 | Rec=0.6341 | F1=0.6767 | Best Epoch=30 | Bes

## Task 4 Documentation Summary

### Retraining After Augmentation
- Dataset loaded: 614 RGB image tiles.
- Split used in the executed notebook: 429 training, 92 validation, and 93 test samples.

| Metric | Latest saved Task 4 run |
|---|---|
| Accuracy | 0.8172 |
| Precision | 0.8769 |
| Recall | 0.6212 |
| F1-Score | 0.6428 |
| Best Epoch | 9 |
| Best Validation Loss | 0.4960 |
| Best Validation Accuracy | 0.7935 |

### CNN Optimization Experiments
- Manual experiment notes indicate the best run used mild geometric augmentation such as random rotation, horizontal flip, and resizing with a 3-layer CNN using 3x3 kernels.
-  Stronger color jitter and vertical flip were associated with the weakest runs, while deeper ReLU-based variants reached the mid-to-high 0.79 accuracy range without beating the best simpler setup.

| Rank | Accuracy | Precision | Recall | F1-Score | Best Epoch | Best Val Loss | Best Val Acc |
|---|---|---|---|---|---|---|---|
| 1 | 0.8065 | 0.8687 | 0.6363 | 0.6774 | 28 | 0.5356 | 0.7283 |
| 2 | 0.8065 | 0.8703 | 0.6341 | 0.6767 | 30 | 0.4682 | 0.7826 |
| 3 | 0.7742 | 0.8478 | 0.6212 | 0.6564 | 27 | 0.5163 | 0.7500 |
| 4 | 0.7957 | 0.5294 | 0.5552 | 0.5419 | 25 | 0.5541 | 0.7391 |
| 5 | 0.7957 | 0.5294 | 0.5552 | 0.5419 | 28 | 0.5333 | 0.7500 |

- 23 of the 27 logged runs still beat the saved Task 3 basic CNN baseline on all four reported metrics.

### Task 3 Results (No Augmentation Data)
| Artifact | Accuracy | Precision | Recall | F1-Score | Notes |
|---|---|---|---|---|---|
| `CNN.py` | 0.6774 | 0.4484 | 0.4654 | 0.4563 | Basic CNN baseline without augmentation. |

- Compared with the saved Task 3 basic CNN baseline, the latest Task 4 run improved accuracy, precision, recall, and F1-score significantly.


### Task 2 Results
#### `SVM.py`
| Script | Accuracy | Precision | Recall | F1-Score | Notes |
|---|---|---|---|---|---|
| `SVM.py` | 0.8645 | 0.8607 | 0.8645 | 0.5600 macro, 0.8600 weighted | Reproduced by running the script with its current defaults; this is a 4-class pixel-wise SVM classifier. |

#### `task2pt2.py`
| Script | Train MSE | Test MSE | Train MAE | Test MAE | Train R2 | Test R2 |
|---|---|---|---|---|---|---|
| `task2pt2.py (NN)` | 167.240451 | 168.042636 | 9.178409 | 9.191253 | 0.7694 | 0.7685 |
